<a href="https://sigmoidal.ai">
  <img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="300">
</a>

# Calibracao de Camera com OpenCV

**Autor:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

Neste notebook, vamos implementar a **calibracao de camera** usando OpenCV e Python.

Usaremos imagens de um tabuleiro de xadrez para detectar cantos, calcular os parametros intrinsecos da camera e corrigir a distorcao das imagens.

In [ ]:
!pip install opencv-python-headless matplotlib -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import urllib.request

## Baixar Imagens de Calibracao

Vamos usar as imagens de tabuleiro de xadrez disponiveis nos samples oficiais do OpenCV.

In [ ]:
# Baixar imagens de calibracao dos samples oficiais do OpenCV
base_url = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/"
image_names = ["left01.jpg", "left02.jpg", "left03.jpg", "left04.jpg",
               "left05.jpg", "left06.jpg", "left07.jpg", "left08.jpg",
               "left09.jpg", "left11.jpg", "left12.jpg", "left13.jpg",
               "left14.jpg"]

os.makedirs("chessboard", exist_ok=True)

for name in image_names:
    url = base_url + name
    filepath = os.path.join("chessboard", name)
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(url, filepath)
        print(f"Baixado: {name}")

print(f"\nTotal de imagens: {len(glob.glob('chessboard/*.jpg'))}")

## Detectar Cantos do Tabuleiro de Xadrez

As imagens do OpenCV samples usam um tabuleiro com **7x6 cantos internos**. Vamos detectar esses cantos em cada imagem.

In [ ]:
# Parametros do tabuleiro de xadrez
chessboard_size = (7, 6)

# Preparar pontos do objeto 3D
objp = np.zeros((np.prod(chessboard_size), 3), dtype=np.float32)
objp[:, :2] = np.indices(chessboard_size).T.reshape(-1, 2)

# Listas para armazenar pontos 3D e pontos 2D
object_points = []
image_points = []

list_of_image_files = sorted(glob.glob('chessboard/*.jpg'))

# Processar cada imagem
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()

for idx, image_file in enumerate(list_of_image_files):
    image = cv2.imread(image_file)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Detectar cantos
    ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)

    if ret:
        object_points.append(objp)
        image_points.append(corners)

        # Desenhar cantos detectados
        cv2.drawChessboardCorners(image, chessboard_size, corners, ret)

    if idx < len(axes):
        axes[idx].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(os.path.basename(image_file), fontsize=9)
        axes[idx].axis('off')

# Esconder eixos vazios
for i in range(len(list_of_image_files), len(axes)):
    axes[i].axis('off')

plt.suptitle('Cantos detectados nas imagens de calibracao', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Cantos detectados em {len(object_points)} de {len(list_of_image_files)} imagens.")

## Calibrar a Camera

Com os pontos 3D e 2D correspondentes, podemos calcular a **matriz intrinseca K** e os **coeficientes de distorcao**.

In [ ]:
# Calibrar a camera
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    object_points, image_points, gray.shape[::-1], None, None
)

print("Matriz de calibracao K:")
print(np.round(K, 2))
print(f"\nCoeficientes de distorcao: {np.round(dist.ravel(), 4)}")
print(f"\nErro de reprojecao (RMS): {ret:.4f}")

## Corrigir Distorcao

Agora vamos usar a matriz K e os coeficientes de distorcao para corrigir uma imagem.

In [ ]:
# Carregar uma imagem de teste
test_image = cv2.imread(list_of_image_files[0])

# Corrigir distorcao
undistorted_image = cv2.undistort(test_image, K, dist, None, K)

# Visualizar antes e depois
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
ax1.set_title('Original (com distorcao)', fontsize=13)
ax1.axis('off')

ax2.imshow(cv2.cvtColor(undistorted_image, cv2.COLOR_BGR2RGB))
ax2.set_title('Corrigida (sem distorcao)', fontsize=13)
ax2.axis('off')

plt.suptitle('Comparacao: antes e depois da calibracao', fontsize=14)
plt.tight_layout()
plt.show()